# 10-工程化：测试、可观测、部署

## 把图从“能跑”变成“能交付”

Tests · Observability · Local server · Deploy · 第十课 · 约 40 分钟

### 目录

- 回顾：你已经会写图了，但还不“敢上线”
- 一、为什么工程化一定要做？
- 二、测试：让图在 CI 里跑得稳
- 三、可观测：出了问题你能定位到哪一步
- 四、本地服务化：用 `langgraph dev` 跑起来
- 五、部署：把图变成一个可调用的 API
- 六、动手跑一下
- 七、检查理解
- 八、常见坑速查
- 九、总结
- 📖 参考（官方）


### 回顾：你已经会写图了，但还不“敢上线”

到第 09 节为止，你已经掌握了：

- 图的控制流（分支/循环/Command）
- 持久化与人工介入（Checkpointer、interrupt/resume）
- 容错（重试/超时/兜底/优雅停止与恢复）

但真正上生产时，你最怕的是三件事：

- **没人保证改代码不把流程搞坏**（缺测试）
- **线上出问题不知道卡在第几步**（缺可观测）
- **图怎么变成“一个服务”让别人调用**（缺部署路径）

这一节就是把这三件事补齐。

---

### 一、为什么工程化一定要做？

你可以把 graph 想象成“一个带状态的业务流程引擎”。

- 代码一改，路由就可能变；
- checkpointer/store 让它跨轮运行；
- streaming / interrupt / retry / timeout 又引入了更多运行时分支。

所以工程化的目标不是“更优雅”，而是三个字：**可控性**。

> 一句话：
>
>- **测试**让行为可控
>- **可观测**让定位可控
>- **部署**让运行可控


---

### 二、测试：让图在 CI 里跑得稳

测试 LangGraph，最常见的坑不是“assert 写错”，而是 **状态被上一次测试污染**。

这通常来自两件事：

- 复用同一个 `thread_id`
- 复用同一个 `checkpointer`（尤其是 InMemorySaver）

下面我们用一个离线可跑的“退款小流程”来演示正确的测试姿势：

- 每个测试用例：**compile 新 graph**（或者至少 new 一个 checkpointer）
- 每个用例：使用 **新的 thread_id**


In [2]:
import time
import uuid
import operator
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


class RefundState(TypedDict):
    amount: int
    logs: Annotated[list[str], operator.add]
    result: str


def now_ms() -> int:
    return int(time.time() * 1000)


def build_refund_graph(*, checkpointer: InMemorySaver | None = None):
    def prepare(state: RefundState) -> dict:
        return {"logs": [f"[{now_ms()}] prepare amount={state['amount']}"]}

    def finalize(state: RefundState) -> dict:
        return {
            "logs": [f"[{now_ms()}] finalize"],
            "result": f"已退款 {state['amount']} 元",
        }

    builder = StateGraph(RefundState)
    builder.add_node("准备", prepare)
    builder.add_node("执行", finalize)
    builder.add_edge(START, "准备")
    builder.add_edge("准备", "执行")
    builder.add_edge("执行", END)

    return builder.compile(checkpointer=checkpointer)


def run_case(amount: int):
    # 单个用例：新 checkpointer + 新 thread_id，保证隔离
    checkpointer = InMemorySaver()
    graph = build_refund_graph(checkpointer=checkpointer)
    thread_id = f"test-{uuid.uuid4().hex[:8]}"
    config = {"configurable": {"thread_id": thread_id}}

    init_state: RefundState = {"amount": amount, "logs": [], "result": ""}
    out = graph.invoke(init_state, config=config)
    return out


out = run_case(500)
out["result"], len(out["logs"])

('已退款 500 元', 2)

In [ ]:
# 在 notebook 里，我们用 assert 模拟最小的“单元测试”
out = run_case(500)
assert out["result"] == "已退款 500 元"
assert out["logs"][0].endswith("prepare amount=500")
assert out["logs"][-1].endswith("finalize")

out2 = run_case(300)
assert out2["result"] == "已退款 300 元"

"tests passed"


> **❓ 检查理解 ①**
>
> 下面哪种做法最容易导致“测试互相污染”？
>
> - A. 每个测试都新建 `thread_id`，但复用同一个 `InMemorySaver`
> - B. 每个测试都复用同一个 `thread_id`，但新建 `InMemorySaver`
> - C. 每个测试都新建 `thread_id` 且新建 `InMemorySaver`
>
> **✅ 答案：A（最常见）**
>
> 解释：`checkpointer + thread_id` 一起决定“读哪个存档”。复用同一个 checkpointer 时，只要某个测试不小心复用了 thread_id（或某个路径写入了共享状态），就会出现非常隐蔽的串味。工程上最稳妥的是：**每个测试新建 checkpointer，并给每个用例新建 thread_id**（也就是 C）。


---

### 三、可观测：出了问题你能定位到哪一步

可观测的目标很朴素：**你能回答“卡在哪一步、为什么”**。

在 notebook/本地开发阶段，你可以先做到两件事：

- **输出可追踪**：把关键决策写入 `logs`（我们前面很多节都这么做了）
- **执行可回放**：遇到问题先用 `stream_mode="updates"/"values"` 看每一步的变化

如果你要上生产，推荐接入 LangSmith Tracing（可选）：

- 通过环境变量打开 tracing
- 用 `tags` / `metadata` 标注 user_id / 环境 / 版本，方便筛选

> 注意：这节默认离线可跑；LangSmith 部分只给“可选开关”，不会让 Run All 失败。


In [ ]:
# 本地可观测（离线可跑）：用 stream 看每步更新

graph = build_refund_graph(checkpointer=InMemorySaver())
thread_id = f"obs-{uuid.uuid4().hex[:8]}"
config = {"configurable": {"thread_id": thread_id}}

init_state: RefundState = {"amount": 500, "logs": [], "result": ""}

print("=== updates ===")
for chunk in graph.stream(init_state, config=config, stream_mode="updates"):
    print(chunk)

print("\n=== values ===")
for i, snapshot in enumerate(graph.stream(init_state, config=config, stream_mode="values")):
    print(f"-- step {i} --")
    print(snapshot)
    print()


> **❓ 检查理解 ②**
>
> 你想在日志/Tracing 里把一次运行标记成“生产环境 + 某个用户”，最合适的放置位置是哪里？
>
> - A. 写死在节点函数里（比如 `logs.append("prod")`）
> - B. 作为 `invoke(..., config={tags, metadata})` 的参数传入
> - C. 写死在 state schema 里（比如加一个字段 `env: Literal["prod"]`）
>
> **✅ 答案：B**
>
> 解释：环境、版本、user_id 这类信息更像“运行上下文”，应该通过 `config` 传入（tags/metadata），而不是污染业务 state，也不应该写死在节点实现里。


---

### 四、本地服务化：用 `langgraph dev` 跑起来

当你想让别人“像调用 API 一样调用你的图”，最小路径通常是：先把图跑成一个本地 Agent Server。

官方推荐做法（摘要）：

1) 安装 CLI（在 conda 环境里直接 pip 安装即可）：

```bash
pip install -U "langgraph-cli[inmem]"
```

2) 启动本地服务：

```bash
langgraph dev
```

启动后你会得到三样东西：

- **API**：例如 `http://127.0.0.1:2024`
- **Studio UI**：用于可视化和调试
- **API Docs**：OpenAPI 文档

> 说明：`langgraph dev` 的 in-memory 模式适合开发/测试；生产需要持久化后端。


---

### 五、部署：把图变成一个可调用的 API

当你准备上线时，核心问题就变成：

- 谁来跑这个图（基础设施）？
- 状态存哪（checkpointer/store 的后端）？
- 怎么灰度、怎么回滚、怎么观测？

官方给的主路径是 LangSmith Cloud 部署：

- 代码在 GitHub 仓库
- 在 LangSmith 里创建 Deployment
- 部署后得到一个 API URL
- 用 SDK 或 REST 调用（支持 streaming）

> 这节只建立“路径认知”，不强制你现在就部署成功（避免把学习卡在账号/权限/网络上）。


---

### 六、动手跑一下

这一节我们把“工程化最小闭环”跑通：

- 用 `assert` 写出可重复的“测试”
- 用 `stream_mode` 观察执行过程

你能做到这两步，就已经比大多数“只会 invoke”更接近可交付了。


In [ ]:
# 1) 最小测试闭环
out = run_case(500)
assert out["result"] == "已退款 500 元"

# 2) 最小可观测闭环（values）
checkpointer = InMemorySaver()
graph = build_refund_graph(checkpointer=checkpointer)
thread_id = f"demo-{uuid.uuid4().hex[:8]}"
config = {"configurable": {"thread_id": thread_id}}

snapshots = list(graph.stream({"amount": 200, "logs": [], "result": ""}, config=config, stream_mode="values"))
assert snapshots[-1]["result"] == "已退款 200 元"

"ok"


---

### 七、检查理解

> **❓ 检查理解 ③**
>
> 你在做“离线优先”的教程/测试时，下面哪种做法最推荐？
>
> - A. 所有 demo 默认调用真实 LLM（这样最贴近生产）
> - B. 默认用纯 Python/固定输出跑通机制，另给“可选真实 LLM” cell
> - C. 默认不提供任何可运行代码，只讲概念（避免环境问题）
>
> **✅ 答案：B**
>
> 解释：学习/测试首先追求“可复现”。真实 LLM 输出不稳定、还依赖 Key/网络；把它放在可选 cell 最合适。


---

### 八、常见坑速查

| 症状 | 原因 | 解法 |
| --- | --- | --- |
| 测试偶尔失败/串味 | 复用 `thread_id` 或复用 `InMemorySaver` | 每个用例新建 `thread_id`，更稳的是每个用例新建 checkpointer |
| 测试很慢、还不稳定 | 默认跑真实 LLM 或外部 API | 测试/教程默认离线 stub，真实调用放可选 cell |
| 线上出了问题不知道卡哪 | 没有可观测输出/没有 tracing | 至少写 `logs`；开发用 `stream_mode`；生产接 LangSmith tracing |
| 本地能跑，上线挂 | 环境变量/依赖/配置不一致 | 把配置放 `.env.example`；部署前用 `langgraph dev` 验证 |

### 九、总结

| 主题 | 一句话 | 你应该记住什么 |
| --- | --- | --- |
| 测试 | 用最小 assert 固化核心路径 | `checkpointer + thread_id` 会影响测试隔离 |
| 可观测 | 让你知道“跑到哪一步了” | 本地先用 `stream_mode`，生产用 tracing（tags/metadata） |
| 本地服务化 | `langgraph dev` 把图跑成 API | in-memory 适合开发；生产要持久化后端 |
| 部署 | 让别人通过 URL 调你的图 | 把“账号/网络/Key”从主线学习里拆出去 |

📖 参考（官方）

- [LangSmith Observability](https://docs.langchain.com/oss/python/langgraph/observability)
- [Run a local server](https://docs.langchain.com/oss/python/langgraph/local-server)
- [Deployment](https://docs.langchain.com/oss/python/langgraph/deploy)
